In [1]:
import unittest
from pathlib import Path

import numpy as np
import pandas as pd

In [2]:
FILES = {
    "Rule-based": "synthetic_students.csv",
    "Seed-based": "my_synthetic_students_data.csv",
}
SEED_FILE = "Synthetic Dataset.csv"
PASS_MARK = 50

COLUMNS = ["ID", "Age", "Study_hrs", "Attendance%", "Total_screen_time",
           "Sleep_hrs", "Internet_quality", "exam_score", "Result"]
NUM = ["Age", "Study_hrs", "Attendance%", "Total_screen_time",
       "Sleep_hrs", "exam_score"]
RANGES = {"Age": (17, 45), "Study_hrs": (0, 40), "Attendance%": (0, 100),
          "Total_screen_time": (0, 24), "Sleep_hrs": (0, 24),
          "exam_score": (0, 100)}

In [3]:
def load_clean(path):
    df = pd.read_csv(path)
    lookup = {c.lower(): c for c in COLUMNS}
    df.columns = [lookup.get(c.strip().replace(" ", "_").lower(), c.strip())
                  for c in df.columns]
    return df

In [4]:
class TestDatasetFiles(unittest.TestCase):
    @classmethod
    def setUpClass(cls):
        cls.data = {name: load_clean(path) for name, path in FILES.items()}
        cls.seed = load_clean(SEED_FILE)

    def test_has_the_nine_columns_in_order(self):
        for name, df in self.data.items():
            with self.subTest(dataset=name):
                self.assertEqual(list(df.columns), COLUMNS)

In [5]:
def test_no_missing_values(self):
        for name, df in self.data.items():
            with self.subTest(dataset=name):
                self.assertEqual(int(df.isna().sum().sum()), 0)


In [6]:
def test_ids_are_unique_and_fake_looking(self):
        for name, df in self.data.items():
            with self.subTest(dataset=name):
                self.assertTrue(df["ID"].is_unique)
                self.assertTrue(df["ID"].astype(str).str.match(r"^S\d{4,}$").all())


In [7]:
 def test_numbers_stay_in_valid_ranges(self):
        for name, df in self.data.items():
            for col, (lo, hi) in RANGES.items():
                with self.subTest(dataset=name, column=col):
                    self.assertTrue(df[col].between(lo, hi).all())


In [8]:
def test_internet_quality_values_are_valid(self):
        words = {"Poor", "Average", "Good"}
        for name, df in self.data.items():
            with self.subTest(dataset=name):
                q = df["Internet_quality"]
                if pd.api.types.is_numeric_dtype(q):
                    self.assertTrue(q.between(1, 5).all())
                else:
                    self.assertTrue(q.astype(str).str.strip().str.title().isin(words).all())

In [9]:
def test_result_matches_the_pass_mark(self):
        for name, df in self.data.items():
            with self.subTest(dataset=name):
                expected = np.where(df["exam_score"] >= PASS_MARK, "Pass", "Fail")
                actual = df["Result"].astype(str).str.strip().str.title().to_numpy()
                self.assertTrue((actual == expected).all())


In [10]:
def test_pass_rate_is_realistic(self):
        for name, df in self.data.items():
            with self.subTest(dataset=name):
                rate = (df["Result"].astype(str).str.strip().str.title() == "Pass").mean()
                self.assertTrue(0.30 <= rate <= 0.90, f"pass rate is {rate:.0%}")

In [11]:
def test_no_duplicate_rows(self):
        for name, df in self.data.items():
            with self.subTest(dataset=name):
                self.assertEqual(int(df.drop(columns=["ID"]).duplicated().sum()), 0)


In [12]:
def test_no_exact_copies_of_seed_rows(self):
        right = self.seed[NUM].astype(float).drop_duplicates()
        for name, df in self.data.items():
            with self.subTest(dataset=name):
                copies = len(df[NUM].astype(float).merge(right, on=NUM, how="inner"))
                self.assertEqual(copies, 0)

In [13]:
def test_no_emails_or_phone_numbers(self):
        pattern = r"[\w.+-]+@[\w-]+\.[\w.-]+|\+?\d[\d\s().-]{8,}\d"
        for name, df in self.data.items():
            with self.subTest(dataset=name):
                text_cols = [c for c in df.columns if not pd.api.types.is_numeric_dtype(df[c])]
                found = sum(int(df[c].astype(str).str.contains(pattern, regex=True).sum())
                            for c in text_cols if c != "ID")
                self.assertEqual(found, 0)

In [14]:
suite = unittest.defaultTestLoader.loadTestsFromTestCase(TestDatasetFiles)
with open("test_results.txt", "w", encoding="utf-8") as f:
    result = unittest.TextTestRunner(stream=f, verbosity=2).run(suite)
    f.write("\nALL TESTS PASSED\n" if result.wasSuccessful() else "\nSOME TESTS FAILED\n")
print(open("test_results.txt", encoding="utf-8").read())

test_has_the_nine_columns_in_order (__main__.TestDatasetFiles.test_has_the_nine_columns_in_order) ... ok

----------------------------------------------------------------------
Ran 1 test in 0.042s

OK

ALL TESTS PASSED

